# Phase 1 — Grid transmission distance

Computes **distance to nearest transmission line (km)** for every 10 km weather grid cell.

| Input | Output |
|-------|--------|
| `data/processed/weather_cell_features.parquet` | `data/processed/grid_transmission_distance.parquet` |
| `data/US_Electric_Power_Transmission_Lin2es_*.geojson` | |

Uses projected CRS **EPSG:32618** (UTM 18N) for meter-accurate distances.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import geopandas as gpd
import polars as pl
import yaml
from shapely.geometry import Point

ROOT = Path('../..').resolve()
CONFIG_PATH = ROOT / 'config' / 'paths.yaml'
WEATHER_PATH = ROOT / 'data' / 'processed' / 'weather_cell_features.parquet'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'grid_transmission_distance.parquet'

WGS84 = 'EPSG:4326'
TARGET_CRS = 'EPSG:32618'
NY_CLIP_BUFFER_M = 50_000  # clip national lines to NY extent + buffer

with CONFIG_PATH.open(encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

transmission_path = ROOT / CONFIG['data']['raw']['transmission_geojson']
if not transmission_path.exists():
    matches = sorted((ROOT / 'data').glob('US_Electric_Power_Transmission*.geojson'))
    if not matches:
        raise FileNotFoundError('Transmission GeoJSON not found under data/')
    transmission_path = matches[0]

if not WEATHER_PATH.exists():
    raise FileNotFoundError(f'Run 01_ingest_weather.ipynb first: {WEATHER_PATH}')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

cells = (
    pl.read_parquet(WEATHER_PATH)
    .select(['latitude', 'longitude'])
    .drop_nulls()
    .unique()
    .sort(['latitude', 'longitude'])
)

print('Cells      :', len(cells))
print('Transmission:', transmission_path)

In [ ]:
cells_pdf = cells.to_pandas()

cells_gdf = gpd.GeoDataFrame(
    cells_pdf,
    geometry=[Point(r.longitude, r.latitude) for r in cells_pdf.itertuples()],
    crs=WGS84,
).to_crs(TARGET_CRS)

lines = gpd.read_file(transmission_path)
if lines.crs is None:
    lines = lines.set_crs(WGS84)
lines = lines[lines.geometry.notna() & ~lines.geometry.is_empty]
lines = lines[lines.geometry.geom_type.isin(['LineString', 'MultiLineString'])].to_crs(TARGET_CRS)

minx, miny, maxx, maxy = cells_gdf.total_bounds
lines_ny = lines.cx[minx - NY_CLIP_BUFFER_M:maxx + NY_CLIP_BUFFER_M, miny - NY_CLIP_BUFFER_M:maxy + NY_CLIP_BUFFER_M]
print(f'Transmission segments (NY clip): {len(lines_ny):,}')

In [ ]:
nearest = gpd.sjoin_nearest(
    cells_gdf,
    lines_ny[['geometry']],
    how='left',
    distance_col='dist_to_transmission_m',
)

out = pl.from_pandas(
    nearest.drop(columns=['geometry', 'index_right'], errors='ignore')[
        ['latitude', 'longitude', 'dist_to_transmission_m']
    ]
).with_columns(
    (pl.col('dist_to_transmission_m') / 1000.0).alias('dist_to_transmission_km')
).sort(['latitude', 'longitude', 'dist_to_transmission_km']).unique(
    subset=['latitude', 'longitude'],
    keep='first',
)

out.write_parquet(OUTPUT_PATH)
print(f'Wrote {len(out):,} rows -> {OUTPUT_PATH}')

In [ ]:
# QA summary
out.select([
    pl.len().alias('n_cells'),
    pl.col('dist_to_transmission_km').mean().alias('mean_km'),
    pl.col('dist_to_transmission_km').median().alias('median_km'),
    pl.col('dist_to_transmission_km').min().alias('min_km'),
    pl.col('dist_to_transmission_km').max().alias('max_km'),
    (pl.col('dist_to_transmission_km') < 2).sum().alias('cells_within_2km'),
    (pl.col('dist_to_transmission_km') > 10).sum().alias('cells_beyond_10km'),
])

In [ ]:
out.sort('dist_to_transmission_km').head(10)